# Pregunta 5 — Desvíos respecto de la receta

**Pregunta de investigación:** ¿Cuánto se apartan los consumos reales de las cantidades de receta y qué costo representa?

**Fuente:** `registro_produccion_LIMPIO (4).xlsx`, hoja `Registro Produccion`. Se utiliza el archivo original **solo en lectura**. Los precios ya fueron divididos por 5 en origen y **no se vuelven a dividir**.

**Objetivo del análisis:** Medir por insumo y masa el costo de las diferencias entre consumo real y receta escalada.

**Indicador:** Indicadores calculados en las celdas siguientes, con unidad y período declarados.


## 1. Contexto
La receta es una referencia; importa cuantificar cuánto se consume de más o de menos.

## 2. Gancho
**¿Qué puede sorprender?** La diferencia entre elaboraciones, grupos o períodos podría ser mayor de lo que sugiere mirar únicamente un promedio general. La magnitud exacta se establece con los datos, no se supone.

## 3. Preparación reproducible
Las transformaciones se calculan en memoria. No se modifica ninguna celda del Excel original.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from artifact_tool import Blob, SpreadsheetFile
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
SEMILLA=42
ruta=Path('registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): ruta=Path('/mnt/data/registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): raise FileNotFoundError('Colocá el Excel junto al notebook')
w=SpreadsheetFile.import_xlsx(Blob.load(str(ruta)))
mat=w.worksheets.get_item_at(0).get_range('A1:CA1112').values
headers=mat[0]; registros=mat[1:]; ix={h:i for i,h in enumerate(headers)}
def valor(r,col):
    x=r[ix[col]]
    try: return float(x) if x is not None and x!='' else np.nan
    except (ValueError,TypeError): return np.nan
def matriz(cols,filas=registros): return np.array([[valor(r,c) for c in cols] for r in filas],float)
def tasas(r):
    l=valor(r,'LITROS DE LECHE')
    return [valor(r,c)*1000/l if np.isfinite(l) and l>0 else np.nan for c in ['AZUCAR','SUERO','BICARBONATO','L.P.D']]
def features():return np.array([[valor(r,'LITROS DE LECHE'),*tasas(r),valor(r,'MATERIA GRASA'),valor(r,'PROTEINAS')] for r in registros],float)
X=features();masas=np.array([str(r[ix['MASA']]) for r in registros]);rend=matriz(['RENDIMIENTO %']).ravel();kg=matriz(['TOTAL KG OBTENIDOS']).ravel()
def normalizar(X):return Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]).fit_transform(X)
def grafico(titulo,unidad,transformacion):
    plt.title(titulo,loc='left',fontweight='bold',fontsize=13)
    plt.figtext(.01,.005,f'Fuente: Registro Produccion | {unidad} | {transformacion} | N={len(registros)}',fontsize=8,color='dimgray')
    plt.tight_layout(rect=(0,.055,1,1));plt.show()
print('Archivo leído en modo solo lectura:',len(registros),'elaboraciones; ',len(headers),'columnas')


## 4. Hallazgos y gráficos
Los gráficos se eligen por la relación que se quiere mostrar: ranking, distribución, comparación, correlación o evolución temporal. Las figuras llevan fuente, unidad y transformación. Los resultados que aparecen a continuación se calculan al ejecutar las celdas.

In [ ]:
pares=[('Azúcar','AZUCAR','AZUCAR según RECETA','COSTO AZUCAR'),('Suero','SUERO','SUERO según RECETA','COSTO SUERO'),('Bicarbonato','BICARBONATO','BICARBONATO según RECETA','COSTO BICARBONATO'),('L.P.D.','L.P.D','L.P.D. según RECETA','COSTO L.P.D.')]
desv=[];cantdesv=[]
for r in registros:
    base=valor(r,'LITROS DE LECHE según RECETA');l=valor(r,'LITROS DE LECHE');factor=l/base if np.isfinite(base) and base>0 else np.nan
    cantidades=[valor(r,real)-valor(r,rec)*factor for _,real,rec,_ in pares]
    cantdesv.append(cantidades);desv.append([d*valor(r,precio) for d,(_,_,_,precio) in zip(cantidades,pares)])
desv=np.array(desv);cantdesv=np.array(cantdesv);nombres=[p[0] for p in pares];suma=np.nansum(desv,axis=0)
fig,ax=plt.subplots(figsize=(9,4));ax.bar(nombres,suma);ax.axhline(0,color='gray');ax.set_ylabel('Desvío económico neto (ajustado)');grafico('Diferencia de costo real menos receta','unidades monetarias ajustadas','receta escalada a litros reales; valores positivos: sobreconsumo')
orden=sorted(set(masas));por_masa=[np.nansum(desv[masas==z])/sum(masas==z) for z in orden]
fig,ax=plt.subplots(figsize=(10,4));ax.bar(orden,por_masa);ax.axhline(0,color='gray');ax.set_ylabel('Desvío neto medio por elaboración');grafico('Desvíos de receta por masa','unidades monetarias ajustadas','suma neta de insumos / elaboraciones de cada masa')
print('Registros válidos por insumo:',{n:int(np.isfinite(desv[:,i]).sum()) for i,n in enumerate(nombres)})
print('COMPRENSIÓN: mayor desvío absoluto neto por insumo:',nombres[np.argmax(np.abs(suma))],'; atención: excesos y ahorros pueden compensarse.')


## 5. Comprensión: insight central
Tomá la frase que imprime la celda anterior como base de la conclusión. **No afirmes causas** que los datos no permitan demostrar. Explicá qué se observa, cuánto representa y qué limitaciones tiene.

## 6. Acción propuesta
Revisar los insumos con mayor desvío económico y validar la base de las recetas.

## 7. Control ético y límites
- Fuente: registro de producción suministrado por la empresa. Período: el cubierto por el Excel; sin recortar meses intencionalmente.
- Indicá unidades, cantidad de registros válidos y transformaciones aplicadas.
- Los gráficos de barras parten de cero; los gráficos de distribución y comparación no ocultan categorías.
- No se cambian ni completan los datos originales; cualquier imputación ocurre únicamente en el modelo.
- La interpretación se limita a asociaciones y predicciones, no a causalidad comprobada.
